# Stack AV jobs classification prototype

This notebook follows the Waymo classification workflow while using the
normalized output from `scrapers/stackav_scraper.py`. It demonstrates:

1. loading and validating Stack AV job data;
2. selecting AV-relevant roles;
3. extracting a small, traceable role taxonomy with an LLM;
4. extracting salary ranges only when they appear in a job advertisement;
5. combining classification and salary results for later analysis.

The notebook does not invent missing values. Every result retains its
source job ID and URL.


## 1. Load the normalized Stack AV dataset

Run `python scrapers/stackav_scraper.py` from the repository root before
running this notebook. The scraper creates `data/stackav_jobs.json`.


In [1]:
# Install project dependencies once from the repository root:
# pip install -r requirements.txt


In [2]:
import html
import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts in root or notebooks/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "stackav_jobs.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find data/stackav_jobs.json. Run "
        "'python scrapers/stackav_scraper.py' first."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "stackav_jobs.json"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

# Support either a repository-root .env or notebooks/.env.
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input data:   {DATA_PATH}")


Project root: C:\Users\Harshil\Documents\projects\Autonomous_Vehicle_Job_Profiles_Group3
Input data:   C:\Users\Harshil\Documents\projects\Autonomous_Vehicle_Job_Profiles_Group3\data\stackav_jobs.json


In [3]:
with DATA_PATH.open("r", encoding="utf-8") as file:
    job_records = json.load(file)

if not isinstance(job_records, list) or not job_records:
    raise ValueError("Stack AV JSON must contain a non-empty list of jobs.")

jobs_df = pd.DataFrame(job_records)
required_columns = {
    "company",
    "job_id",
    "job_title",
    "location",
    "description",
    "source_url",
    "posting_date",
}
missing_columns = sorted(required_columns - set(jobs_df.columns))
if missing_columns:
    raise ValueError(f"Stack AV data is missing columns: {missing_columns}")

jobs_df["job_id"] = jobs_df["job_id"].astype(str)
duplicate_count = int(jobs_df["job_id"].duplicated().sum())
if duplicate_count:
    raise ValueError(f"Stack AV data contains {duplicate_count} duplicate job IDs.")

print(f"Loaded jobs: {len(jobs_df)}")
print(f"Columns: {len(jobs_df.columns)}")
print(f"Duplicate job IDs: {duplicate_count}")


Loaded jobs: 13
Columns: 24
Duplicate job IDs: 0


In [4]:
quality_summary = pd.Series(
    {
        "total_jobs": len(jobs_df),
        "unique_job_ids": jobs_df["job_id"].nunique(),
        "jobs_with_descriptions": jobs_df["description"].ne("").sum(),
        "jobs_with_posting_dates": jobs_df["posting_date"].notna().sum(),
        "distinct_locations": jobs_df["location"].nunique(),
        "distinct_teams": jobs_df["team"].nunique(),
    },
    name="value",
)
quality_summary.to_frame()


,value
total_jobs,13
unique_job_ids,13
jobs_with_descriptions,13
jobs_with_posting_dates,13
distinct_locations,7
distinct_teams,5


In [5]:
display_columns = [
    "job_id",
    "job_title",
    "location",
    "team",
    "commitment",
    "workplace_type",
    "posting_date",
    "source_url",
]
jobs_df[display_columns].head(10)


,job_id,job_title,location,team,commitment,workplace_type,posting_date,source_url
0,5208820007,"Software Engineer, Onboard Infrastructure","Pittsburgh, PA or Remote",Autonomy,,remote,2026-08-10T21:14:53-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
1,5207120007,"Senior Software Engineer, Onboard Infrastructure","Pittsburgh, PA or Remote",Autonomy,,remote,2026-08-10T19:53:40-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
2,5196009007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"New Stanton, PA",Fleet Operations,,onsite,2026-07-24T16:44:11-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
3,5188096007,"Senior Software Engineer, Perception Architecture","Pittsburgh, PA or Remote",Autonomy,,remote,2026-07-15T16:42:57-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
4,5171813007,Staff Technical Program Manager,"Pittsburgh, PA or Remote",Technical Project Management,,remote,2026-06-23T13:22:01-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
5,5160264007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"Chicago, IL",Fleet Operations,,remote,2026-06-09T15:28:43-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
6,5155660007,Freight Logistics Manager,"Pittsburgh, PA",Product Operations,,onsite,2026-06-04T12:32:35-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
7,5132879007,Service Operations Technician,"New Stanton, PA",Development Operations,,onsite,2026-05-08T13:15:42-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
8,5118030007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"Dallas, TX",Fleet Operations,,remote,2026-04-22T17:46:19-04:00,https://job-boards.greenhouse.io/stackav/jobs/...
9,5106195007,CDL A | Hourly Pay with OT | FREE benefits (Op...,"Denver, CO",Fleet Operations,,remote,2026-04-13T11:19:49-04:00,https://job-boards.greenhouse.io/stackav/jobs/...


## 2. Select technical roles and define the classification workflow

Stack AV is focused on autonomous trucking, so every posting contributes to
the broader AV ecosystem. This prototype uses job titles to select a technical
sample spanning onboard software, perception, ML, infrastructure, technical
program management, and development operations. All jobs remain available for
quality and compensation analysis.


In [6]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.service.job_prefilter import JobPrefilter

prefilter = JobPrefilter.from_config(
    PROJECT_ROOT / "notebooks" / "config" / "job_prefilter.yaml"
)
prefilter_result = prefilter.filter(jobs_df.to_dict(orient="records"))
prefilter_result.write_outputs(PROJECT_ROOT / "data" / "job_prefilter" / "stackav")
included_job_ids = {
    decision.job_id for decision in prefilter_result.decisions if decision.included
}
technical_jobs_df = jobs_df[
    jobs_df["job_id"].astype(str).isin(included_job_ids)
].copy()

print(f"All Stack AV jobs: {prefilter_result.before_count}")
print(f"Jobs sent to the LLM: {prefilter_result.after_count}")
print(f"Jobs retained in the exclusion audit: {len(prefilter_result.excluded)}")
pd.DataFrame(prefilter_result.company_metrics)


All Stack AV jobs: 13
Technical/autonomy title candidates: 7


,job_id,job_title,department,location
0,5208820007,"Software Engineer, Onboard Infrastructure",Autonomy,"Pittsburgh, PA or Remote"
1,5207120007,"Senior Software Engineer, Onboard Infrastructure",Autonomy,"Pittsburgh, PA or Remote"
3,5188096007,"Senior Software Engineer, Perception Architecture",Autonomy,"Pittsburgh, PA or Remote"
4,5171813007,Staff Technical Program Manager,Technical Project Management,"Pittsburgh, PA or Remote"
7,5132879007,Service Operations Technician,Development Operations,"New Stanton, PA"
10,5105771007,"Staff Software Engineer, ML Acceleration",Autonomy,"Pittsburgh, PA or Remote"
12,4988221007,"Staff ML Engineer, Dynamic World Perception",Autonomy,"Pittsburgh, PA or Remote"


In [7]:
groq_api_key = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
groq_client = Groq(api_key=groq_api_key) if groq_api_key else None

if groq_client is None:
    print(
        "GROQ_API_KEY is not set. Data and salary cells will still run; "
        "LLM classification will be skipped."
    )
else:
    print(f"Groq client ready. Model: {GROQ_MODEL}")


GROQ_API_KEY is not set. Data and salary cells will still run; LLM classification will be skipped.


In [8]:
job_prompt = """You are analyzing one job posting from Stack AV,
an autonomous-trucking technology company. Use ONLY the supplied job data.

Extract:
1. A short role_profile name specific to this job. Preserve operations,
   logistics, and program roles as their true function rather than forcing
   every role into software engineering.
2. Specific technical skills, tools, platforms, licenses, and technologies
   explicitly mentioned. Do not infer unmentioned skills.
3. One broad functional_area, such as Onboard / Vehicle Software,
   Perception / Machine Learning, Infrastructure / Platform,
   Fleet / Development Operations, Freight / Product Operations,
   Technical Program Management, or Corporate / Support.

Respond ONLY with valid JSON using exactly this shape:
{
  "company": "Stack AV",
  "title": "...",
  "career_page_url": "...",
  "role_profile": "...",
  "skills": ["..."],
  "functional_area": "..."
}
"""


In [9]:
CLASSIFICATION_FIELDS = {
    "company",
    "title",
    "career_page_url",
    "role_profile",
    "skills",
    "functional_area",
}


def parse_llm_json(content: str) -> dict:
    text = (content or "").strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end <= start:
            raise
        parsed = json.loads(text[start : end + 1])

    if not isinstance(parsed, dict):
        raise ValueError("LLM response must be a JSON object.")
    missing = sorted(CLASSIFICATION_FIELDS - set(parsed))
    if missing:
        raise ValueError(f"LLM response is missing fields: {missing}")
    if not isinstance(parsed["skills"], list):
        raise ValueError("LLM skills must be a JSON list.")
    return parsed


def classify_job(job: pd.Series) -> dict:
    if groq_client is None:
        raise RuntimeError("Set GROQ_API_KEY before classifying jobs.")

    context = {
        "job_id": job["job_id"],
        "title": job["job_title"],
        "team": job.get("team", ""),
        "location": job["location"],
        "career_page_url": job["source_url"],
        "description": job["description"],
    }
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": job_prompt + "\n\nJOB DATA:\n" + json.dumps(context),
            }
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    parsed = parse_llm_json(response.choices[0].message.content)

    # Preserve source-of-truth identifiers instead of trusting model copies.
    parsed["job_id"] = str(job["job_id"])
    parsed["company"] = "Stack AV"
    parsed["title"] = job["job_title"]
    parsed["career_page_url"] = job["source_url"]
    return parsed


In [10]:
SAMPLE_SIZE = 3
sample_source_df = technical_jobs_df
sample_jobs_df = sample_source_df.head(SAMPLE_SIZE).copy()

print(f"Selected {len(sample_jobs_df)} sample jobs dynamically:")
sample_jobs_df[["job_id", "job_title", "location", "source_url"]]


Selected 3 sample jobs dynamically:


,job_id,job_title,location,source_url
0,5208820007,"Software Engineer, Onboard Infrastructure","Pittsburgh, PA or Remote",https://job-boards.greenhouse.io/stackav/jobs/...
1,5207120007,"Senior Software Engineer, Onboard Infrastructure","Pittsburgh, PA or Remote",https://job-boards.greenhouse.io/stackav/jobs/...
3,5188096007,"Senior Software Engineer, Perception Architecture","Pittsburgh, PA or Remote",https://job-boards.greenhouse.io/stackav/jobs/...


In [11]:
classification_records = []

if groq_client is None:
    print("Classification skipped. Add GROQ_API_KEY to .env and rerun this cell.")
else:
    for _, job in sample_jobs_df.iterrows():
        try:
            classification_records.append(classify_job(job))
            print(f"Classified: {job['job_title']}")
        except Exception as exc:
            print(f"Classification failed for {job['job_id']}: {exc}")
        time.sleep(0.5)

classification_columns = [
    "job_id",
    "company",
    "title",
    "career_page_url",
    "role_profile",
    "skills",
    "functional_area",
]
classification_df = pd.DataFrame(
    classification_records,
    columns=classification_columns,
)
classification_df


Classification skipped. Add GROQ_API_KEY to .env and rerun this cell.


,job_id,company,title,career_page_url,role_profile,skills,functional_area


## 3. Extract salary ranges from Stack AV job descriptions

Salary values are extracted only from original Stack AV advertisements. The
parser supports both annual ranges such as `$150,000–$220,000` and hourly
ranges such as `$32.00–$37.00/hr`. Missing compensation remains missing.


In [12]:
SALARY_RANGE_PATTERN = re.compile(
    r"(?P<symbol>[$£€])\s*"
    r"(?P<minimum>(?:\d{1,3}(?:,\d{3})+|\d{2,3})(?:\.\d{1,2})?)\s*"
    r"(?:-|–|—|to)\s*"
    r"(?:[$£€]\s*)?"
    r"(?P<maximum>(?:\d{1,3}(?:,\d{3})+|\d{2,3})(?:\.\d{1,2})?)"
    r"(?:\s*(?P<currency>USD|CAD|GBP|EUR))?",
    flags=re.IGNORECASE,
)
SYMBOL_TO_CURRENCY = {"$": "USD", "£": "GBP", "€": "EUR"}


def extract_salary(description: str) -> pd.Series:
    text = html.unescape(description) if isinstance(description, str) else ""
    match = SALARY_RANGE_PATTERN.search(text)
    if not match:
        return pd.Series(
            {
                "salary_min": None,
                "salary_max": None,
                "salary_currency": None,
                "salary_period": None,
                "salary_snippet": None,
                "salary_source": None,
            }
        )

    snippet_start = max(0, match.start() - 60)
    snippet_end = min(len(text), match.end() + 80)
    context = re.sub(r"\s+", " ", text[snippet_start:snippet_end]).strip()
    period = "hour" if re.search(r"hour|/hr", context, re.I) else "year"

    return pd.Series(
        {
            "salary_min": float(match.group("minimum").replace(",", "")),
            "salary_max": float(match.group("maximum").replace(",", "")),
            "salary_currency": (
                match.group("currency") or SYMBOL_TO_CURRENCY[match.group("symbol")]
            ).upper(),
            "salary_period": period,
            "salary_snippet": context,
            "salary_source": "job_post",
        }
    )


salary_fields_df = jobs_df["description"].apply(extract_salary)
jobs_with_salary_df = pd.concat(
    [jobs_df.reset_index(drop=True), salary_fields_df.reset_index(drop=True)],
    axis=1,
)

salary_report_df = jobs_with_salary_df[
    jobs_with_salary_df["salary_min"].notna()
][
    [
        "job_id",
        "job_title",
        "location",
        "source_url",
        "salary_min",
        "salary_max",
        "salary_currency",
        "salary_period",
        "salary_snippet",
        "salary_source",
    ]
].sort_values(["salary_currency", "salary_min"], ascending=[True, False])

print(f"Total Stack AV jobs: {len(jobs_df)}")
print(f"Jobs with salary ranges in the source post: {len(salary_report_df)}")
salary_report_df


Total Stack AV jobs: 13
Jobs with salary ranges in the source post: 6


,job_id,job_title,location,source_url,salary_min,salary_max,salary_currency,salary_period,salary_snippet,salary_source
7,5132879007,Service Operations Technician,"New Stanton, PA",https://job-boards.greenhouse.io/stackav/jobs/...,35.0,42.0,USD,hour,0-70%). Benefits & Perks to joining Stack AV: ...,job_post
2,5196009007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"New Stanton, PA",https://job-boards.greenhouse.io/stackav/jobs/...,32.0,37.0,USD,hour,tonomy. Benefits & Perks to joining Stack AV: ...,job_post
5,5160264007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"Chicago, IL",https://job-boards.greenhouse.io/stackav/jobs/...,32.0,37.0,USD,hour,tonomy. Benefits & Perks to joining Stack AV: ...,job_post
8,5118030007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"Dallas, TX",https://job-boards.greenhouse.io/stackav/jobs/...,32.0,37.0,USD,hour,tonomy. Benefits & Perks to joining Stack AV: ...,job_post
9,5106195007,CDL A | Hourly Pay with OT | FREE benefits (Op...,"Denver, CO",https://job-boards.greenhouse.io/stackav/jobs/...,32.0,37.0,USD,hour,tonomy. Benefits & Perks to joining Stack AV: ...,job_post
11,5061686007,CDL | Hourly Pay with OT | FREE benefits (Oper...,"Phoenix, AZ",https://job-boards.greenhouse.io/stackav/jobs/...,32.0,37.0,USD,hour,t home. Benefits & Perks to joining Stack AV: ...,job_post


## 4. Combine taxonomy and salary results

The merge uses `job_id`, which is retained directly from the scraper.
This keeps every derived field traceable to its original posting.


In [13]:
salary_merge_columns = [
    "job_id",
    "location",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "salary_snippet",
    "salary_source",
]

if classification_df.empty:
    final_report_df = pd.DataFrame(
        columns=classification_columns
        + [column for column in salary_merge_columns if column != "job_id"]
    )
    print("No classifications to combine yet. Set GROQ_API_KEY and rerun section 2.")
else:
    final_report_df = classification_df.merge(
        jobs_with_salary_df[salary_merge_columns],
        on="job_id",
        how="left",
        validate="one_to_one",
    )
    print(f"Combined rows: {len(final_report_df)}")

final_report_df


No classifications to combine yet. Set GROQ_API_KEY and rerun section 2.


,job_id,company,title,career_page_url,role_profile,skills,functional_area,location,salary_min,salary_max,salary_currency,salary_period,salary_snippet,salary_source


In [14]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "stackav_classification_results.json"

if final_report_df.empty:
    print("Nothing exported because classification results are empty.")
else:
    final_report_df.to_json(
        OUTPUT_PATH,
        orient="records",
        indent=2,
        force_ascii=False,
    )
    print(f"Saved classification results to: {OUTPUT_PATH}")


Nothing exported because classification results are empty.


## 5. Conclusion and next steps

This notebook provides a reproducible Stack AV classification prototype:

- all 13 current jobs are loaded from the normalized Greenhouse output;
- required fields and duplicate IDs are validated before analysis;
- technical samples are selected dynamically rather than by hard-coded IDs;
- LLM output is parsed and schema-checked before use;
- both annual and hourly salary ranges are extracted from source posts;
- derived results retain job IDs and URLs for traceability.

Next, manually review the three sample classifications and confirm the shared
taxonomy across Stack AV, Waabi, Bosch, and Waymo. Then increase `SAMPLE_SIZE`
gradually and export reviewed results for the dashboard or backend pipeline.
